# 第三章：马尔科夫决策过程 （Finite Markov Decision Processes）  

## 介绍（Introduction）

在本次实验中，熟练掌握马尔可夫过程、马尔可夫奖励过程和马尔可夫决策过程。

***
# 涉及核心概念回顾： 参考课堂上ppt
***
# 评分标准如下：

- [测试1-1 （10分）](#1)
- [测试1-2（10分）](#2)
- [测试1-3（10分）](#3)
- [测试1-4 (10分)](#4)
- [测试1-5 （10分）](#4)
- [测试2-1 （10分）](#1)
- [测试2-2（10分）](#2)
- [测试2-3（10分）](#3)
- [测试2-4 （10分）](#4)
- [测试2-5 （10分）](#4)
***

<div class="alert alert-block alert-warning">

**主题1： 马尔可夫奖励过程（Markov Reward Processes）**
</div>

In [1]:
# 导入需要使用的库
import random 
import numpy as np
random.seed(1234)
np.random.seed(1234)



**测试1-1**：阅读下图状态转移图，写出状态转移概率矩阵和奖励函数（向量表示）。

<img src="./MRP_graph.jpg" style="zoom:50%" />

**Hint 1**: 补充状态空间定义。仿照第一个状态名称，替换*为对应名称

In [2]:
# 用字符串表示每个状态
STATE_SPACE=["s_1", "s_2", "s_3", "s_4", "s_5", "s_6"]

**Hint 2**: 按照上图，定义状态转移概率矩阵

In [3]:
# TODO: 基于上图，替换下面x为所需要的概率数值
P = [
    [0.9, 0.1, 0.0, 0.0, 0.0, 0.0],
    [0.5, 0.0, 0.5, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.6, 0.0, 0.4],
    [0.0, 0.0, 0.0, 0.0, 0.3, 0.7],
    [0.0, 0.2, 0.3, 0.5, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
]
# 对上述二位数组用矩阵进行封装
P = np.array(P)
# 定义奖励函数R
# TODO: 基于上图，替换下面*为所需要的奖励数值
R = {"s_1": -1, "s_2": -2, "s_3": -2, "s_4": 10, "s_5": 1, "s_6": 0}
# 定义折扣因子gamma, 请设置*为一个0~1的数值
gamma = 0.5
 # 用四元组封装MRP (包含gamma)
MRP = (STATE_SPACE, P, R, gamma)

**测试1-1 反思小问**
1. 你在写转移矩阵时最容易出错的一行是哪一行？请说明你如何自查。

最容易出错的是第5行（s_5 对应那一行），因为它有三个非零转移概率，容易把数值位置写反或漏写。

自查方法是：先逐项对照状态转移图核对“从哪个状态出发到哪个状态”，再检查每一行概率和是否等于1，最后确认终止状态 s_6 只有自环概率1。

**测试1-2**： 基于上述状态概率转移矩阵，编写一个函数，采样相应轨迹(episode)

要求：对下述函数填写注释

In [4]:
def sample_trajectory(start_state, probabilities, max_len=1000):
    """
    从给定起始状态出发，按状态转移概率逐步采样后续状态，直到到达终止状态或达到最大长度。
    返回值为状态编号序列（从1开始计数）。
    """
    # 初始化轨迹列表，先放入起始状态（状态编号+1便于和s_1...s_6对应）
    episode = [start_state + 1]   
    # 用state记录当前所在状态（内部仍采用0开始的索引）
    state = start_state    
    # 当未到终止状态s_6且轨迹长度未超过max_len时，持续采样
    while state != 5 and len(episode) < max_len:   
        # 按当前状态对应的转移概率分布随机采样下一个状态索引
        next_state = random.choices(range(len(probabilities[state])), probabilities[state])[0]
        # 将采样得到的下一个状态加入轨迹（转换为1开始编号）
        episode.append(next_state + 1)    
        # 更新当前状态，进入下一轮采样
        state = next_state   
    return episode

**测试1-2 反思小问**
1. 你在 `sample_trajectory` 中若去掉长度上限，可能出现什么问题？

如果状态转移里存在环（例如几个状态之间反复跳转），轨迹可能长时间不终止，甚至造成死循环，导致程序一直运行、资源占用增大。

2. 结合一次你自己的采样输出，解释“随机性”在轨迹中的具体体现。

例如我从 `s_2` 出发做两次采样，得到的轨迹可能分别是 `s_2→s_1→s_2→s_3→s_6` 和 `s_2→s_3→s_4→s_6`。起点相同但中间路径不同，说明每一步都是按概率随机选择，而不是固定路线。

**测试1-3**：
基于状态转移概率，采样轨迹（其实你可以采样无数条轨迹。）
要求：从$s_2$状态为起始状态，采样轨迹长度不能超过5

In [5]:
_, P, _, _ = MRP
# 调用sample_trajectory，从状态s2开始采样一个轨迹，最大长度为5
# TODO: 请注意，这里的状态是从0开始编号的，所以s2对应的是状态1
episode = sample_trajectory(start_state=1, probabilities=P, max_len=5)

# TODO: 打印采样的轨迹
print("从s_2开始采样得到的轨迹为:", episode)

从s_2开始采样得到的轨迹为: [2, 3, 4, 5, 4]


**测试1-4**： 针对上述采样的轨迹，计算出初始状态的回报

基于上述轨迹，计算$s_2$的回报（Return）
提示：$G_t = R_{t+1} + \gamma G_{t+1}$

In [6]:
# 给定一条序列,计算从某个索引（起始状态）开始到序列最后（终止状态）得到的回报
def compute_return(start_index, episode, gamma):
    G = 0
    for i in reversed(range(start_index, len(episode))):
        G = gamma * G + R['s'+'_'+str((episode[i]))]
    return G

# 替换状态s_2为对应的索引
start_index = 0
# 注意：episode是上面采样的轨迹
G = compute_return(start_index, episode, gamma)
print("根据本序列计算得到回报为：%s。" % G)


根据本序列计算得到回报为：0.25。


**测试1-4 反思小问**
1. 当 `gamma` 增大时，你这条轨迹的回报数值会更接近即时奖励还是长期奖励？结合结果说明。

回答：当 `gamma` 增大时，回报会更重视后续奖励，因此更接近长期奖励。以这条轨迹为例，后面若经过高奖励状态（如 `s_4`），更大的 `gamma` 会让这些后续奖励在总回报中占比更高。

2. 其实上述轨迹，有的状态可能会被visit多次，如何计算对应状态的回报。

回答：可以按“每次访问”分别计算该状态从当前时刻起的回报（Every-visit），也可以只用该状态在一条轨迹中的第一次访问来计算（First-visit），再对多条轨迹取平均作为状态价值估计。


**测试1-5**：MRP状态价值函数计算

- 一个状态的期望回报，即从这个状态出发，未来累积奖励的期望。
- 所有状态的价值组成价值函数。
</div>
我们将状态价值函数写成

$$\mathbf{v}(s)=\mathbb{E}[G_t | S_t = s] = \mathbb{E}[R_t + \gamma G_{t+1} | S_t = s]$$

基于贝尔曼方程直接计算状态价值
$$\mathbf{v}(s)= R(s) + \gamma \sum_{s' \in \mathcal{S}} p(s'|s)\mathbf{v}(s') $$

提示：
$$\mathbf{v} = R + \gamma P\mathbf{v}$$
$$(I - \gamma P)\mathbf{v} = R$$
$$\mathbf{v} = (I - \gamma P)^{-1}R$$

基于上述贝尔曼方程，计算MRP状态价值函数。

In [7]:
# TODO: 请在下面编写代码，计算状态价值函数
def compute_state_value(P, R, gamma, STATE_SPACE):
    """
    计算MDP中的状态价值函数
    """
    # 提示：可以使用np.linalg.inv()来计算矩阵的逆，使用np.eye()来生成单位矩阵
    # np.dot()来计算矩阵乘法
    # TODO: 生成单位矩阵
    I = np.eye(len(STATE_SPACE))
    # TODO: 计算价值函数，使用公式V = (I - gamma * P)^(-1) * R
    R_vec = np.array([R[s] for s in STATE_SPACE]).reshape(-1, 1)
    V = np.dot(np.linalg.inv(I - gamma * P), R_vec)
    return {s: float(V[i, 0]) for i, s in enumerate(STATE_SPACE)}

STATE_SPACE,P, R, gamma = MRP
value = compute_state_value(P, R, gamma, STATE_SPACE)
# STEP 5: 输出解析解value
print("MRP中每个状态价值分别为\n", value)
# 输出结果如下
# MRP中每个状态价值分别为
#  [[-2.01950168]
#  [-2.21451846]
#  [ 1.16142785]
#  [10.53809283]
#  [ 3.58728554]
#  [ 0.        ]]

MRP中每个状态价值分别为
 {'s_1': -2.0195016779238815, 's_2': -2.214518457162695, 's_3': 1.161427849273102, 's_4': 10.538092830910342, 's_5': 3.587285539402281, 's_6': 0.0}


**测试1-5 反思小问** 请比较一个状态在“采样估计回报”和“解析解状态价值”的优缺点。

采样估计回报的优点是实现简单、只依赖与环境交互得到的轨迹，不需要已知完整转移概率；缺点是方差较大，样本少时结果不稳定，需要较多采样才能收敛。

解析解状态价值的优点是结果精确、一次求解即可得到全部状态价值；缺点是必须已知模型（转移概率和奖励），且状态空间大时矩阵求逆计算开销高。

<div class="alert alert-block alert-warning">

**主题2： 有限马尔可夫决策过程（finite Markov Decision Processes）**
</div>

以下gridWorld定义的MDP环境。

<img src="./gridWorld.jpg" style="zoom:50%" />

基于上述Grid信息，定义MDP环境（为了方便理解，我们假设奖励本身为确定量，非随机变量）：
$$<\mathcal{S}, \mathcal{A}, \mathcal{P}, R, \gamma>$$
- $\mathcal{S}$: 状态空间
- $\mathcal{A}$: 动作空间
- $\gamma$折扣因子
- $R(s, a)$: 奖励函数，即奖励取决于状态$s$和动作$a$
- $\mathcal{P}(s'|s, a)$为状态转移函数，为在状态$s$下执行动作$a$后到达$s'$的概率。

**测试2-1**： 基于上图，定义马尔可夫决策过程。

In [8]:
# TODO: 定义状态空间("s_A", "s_B", "s_C", "s_D")和动作空间("Up", "Down", "Left", "Right")
# 替换__TODO__为所需要的代码
ACTION_SPACE = ["Up", "Down", "Left", "Right"]
STATE_SPACE = ["s_A", "s_B", "s_C", "s_D"]

P = {f"{s}-{a}-{next_s}": 0.0 for s in STATE_SPACE for a in ACTION_SPACE for next_s in STATE_SPACE}
# 设置给定位置为相应的概率
# s_A: 左上角，上/左边界回弹，右到B，下到C
P["s_A-Up-s_A"] = 1.0
P["s_A-Down-s_C"] = 1.0
P["s_A-Left-s_A"] = 1.0
P["s_A-Right-s_B"] = 1.0
# TODO: 设置其他位置的概率
# s_B: 右上角，上/右边界回弹，左到A，下到D
P["s_B-Up-s_B"] = 1.0
P["s_B-Down-s_D"] = 1.0
P["s_B-Left-s_A"] = 1.0
P["s_B-Right-s_B"] = 1.0
# s_C: 左下角，下/左边界回弹，上到A，右到D
P["s_C-Up-s_A"] = 1.0
P["s_C-Down-s_C"] = 1.0
P["s_C-Left-s_C"] = 1.0
P["s_C-Right-s_D"] = 1.0
# s_D: 右下角，下/右边界回弹，上到B，左到C
P["s_D-Up-s_B"] = 1.0
P["s_D-Down-s_D"] = 1.0
P["s_D-Left-s_C"] = 1.0
P["s_D-Right-s_D"] = 1.0

# TODO: 定义奖励函数, 替换__TODO__为所需要的奖励数值
# 根据图示：A向右到B得+5，其他+0；B所有方向均+5；C/D均为+0，D向上到B得+5
rewards = {
    "s_A": {"Up": 0, "Down": 0, "Left": 0, "Right": 5},
    "s_B": {"Up": 5, "Down": 0, "Left": 0, "Right": 5},
    "s_C": {"Up": 0, "Down": 0, "Left": 0, "Right": 0},
    "s_D": {"Up": 5, "Down": 0, "Left": 0, "Right": 0},
}
# 折扣因子为0.7
gamma = 0.7
# 定义MDP
MDP = {
    "STATE_SPACE": STATE_SPACE,
    "ACTION_SPACE": ACTION_SPACE,
    "P": P,
    "rewards": rewards,
    "gamma": gamma
}

**测试2-2**： 定义一个随机策略$\pi(a|s)$，在上述环境中每个cell上, 可执行的动作为{上，下，左，右}，概率分别为25%。

In [9]:
# 可用字典定义随机策略
# TODO: 请替换下面的None为所需要的代码
policy = {
    "s_A": {"Up": 0.25, "Down": 0.25, "Left": 0.25, "Right": 0.25},
    "s_B": {"Up": 0.25, "Down": 0.25, "Left": 0.25, "Right": 0.25},
    "s_C": {"Up": 0.25, "Down": 0.25, "Left": 0.25, "Right": 0.25},
    "s_D": {"Up": 0.25, "Down": 0.25, "Left": 0.25, "Right": 0.25},
}

**测试2-3**： 基于上述策略，请计算MDP中每个状态的奖励。

我们可以将策略的动作选择进行边缘化（marginalization)，就可以得到没有动作的 MRP 了。具体来说，对于某一个状态，我们根据策略所有动作的概率进行加权，得到的奖励和就可以认为是一个 MRP 在该状态下的奖励，即

$$
R(s) = \sum_{a \in \mathcal{A}} \pi(a|s) R(s, a)
$$

In [10]:
def compute_reward(MDP):
    """
    计算MDP中的奖励函数
    """
    # TODO: 请在下面编写代码
    # 将__TODO__替换为所需要的代码
    STATE_SPACE = MDP["STATE_SPACE"]
    ACTION_SPACE = MDP["ACTION_SPACE"]
    R = MDP["rewards"]
    R_state = {s: 0 for s in STATE_SPACE}
    for state in STATE_SPACE:
        for i in ACTION_SPACE:
            # TODO: 计算状态奖励函数
            # R(s) = sum_{a} pi(a|s) * R(s, a)
            R_state[state] += policy[state][i] * R[state][i]

    return R_state
# 计算奖励函数
R_MRP = compute_reward(MDP)
print("reward: ", R_MRP)

reward:  {'s_A': 1.25, 's_B': 2.5, 's_C': 0.0, 's_D': 1.25}


**测试2-4**： 请计算状态转移概率

我们计算采取动作的概率与使s转移到的s'概率的乘积，再将这些乘积相加，其和就是一个 MRP 的状态s从转移至s'的概率

$$
\mathcal{P}'(s, s') = \sum_{a \in \mathcal{A}} \pi(a|s) P(s'|s, a)
$$

In [11]:
# 状态转移矩阵
def cal_state_transition(MDP):
    """
    计算MDP中的状态转移矩阵
    """
    # TODO: 请在下面编写代码
    # 将None替换为所需要的代码
    STATE_SPACE = MDP["STATE_SPACE"]
    ACTION_SPACE = MDP["ACTION_SPACE"]
    P = MDP["P"]
    # 初始化状态转移矩阵
    P_prim = np.zeros((len(STATE_SPACE), len(STATE_SPACE)))
    for i, s in enumerate(STATE_SPACE):
        for j, next_s in enumerate(STATE_SPACE):
            for a in ACTION_SPACE:
                # TODO: 计算状态转移概率
                # 将None替换为所需要的代码
                # 基于公式P(s'|s, a) = sum(pi(a|s) * P(s'|s, a))
                P_prim[i, j] += policy[s][a] * P[f"{s}-{a}-{next_s}"]
    return P_prim

# 调用cal_state_transition函数，计算状态转移矩阵
# 替换__TODO__为所需要的代码
P_MRP = cal_state_transition(MDP)
print(P_MRP)

[[0.5  0.25 0.25 0.  ]
 [0.25 0.5  0.   0.25]
 [0.25 0.   0.5  0.25]
 [0.   0.25 0.25 0.5 ]]


**测试2-5**： 我们通过对动作进行margalization, 实现了将MDP转换成MRP，现在请调用函数，计算MDP的状态价值函数。

In [12]:
# 计算状态价值函数
# TODO: 请在下面编写代码
# Hint: 用到MRP中的函数

# Step1: 将MDP通过marginalization转化为MRP的奖励函数和转移矩阵
R_MDP = compute_reward(MDP)           # {s: R(s)}字典
P_MDP = cal_state_transition(MDP)     # 4x4 numpy数组

# Step2: 复用测试1-5中的compute_state_value函数计算价值
STATE_SPACE_MDP = MDP["STATE_SPACE"]
gamma_MDP = MDP["gamma"]
value_MDP = compute_state_value(P_MDP, R_MDP, gamma_MDP, STATE_SPACE_MDP)
print("MDP中每个状态价值分别为\n", value_MDP)

MDP中每个状态价值分别为
 {'s_A': 4.166666666666665, 's_B': 6.0897435897435885, 's_C': 2.2435897435897427, 's_D': 4.166666666666666}


**测试2-5 反思小问**
1. 在“MDP 转 MRP 再求价值”的链路中，你认为最关键的一个中间量是什么？

回答：最关键的中间量是“边缘化后的状态转移矩阵 $P'(s, s')$”。它把策略信息吸收进去，将带动作的 MDP 化简为可用贝尔曼方程直接求解的 MRP。转移矩阵计算出错，后续价值必然偶差，因此它是整个链路中最关键的环节。

## 总结
请写一下你对马尔可夫过程、马尔可夫奖励过程，马尔可夫决策过程的异同点。

三者的共同点在于都满足马尔可夫性质：下一个状态仅与当前状态有关，与过去历史无关。它们都具有状态空间和状态转移概率的核心结构。

差异点如下：

马尔可夫过程（MP）是最基础的模型，只描述状态如何按概率转移，无奖励、无决策者，无法评价状态的好坏。

马尔可夫奖励过程（MRP）在MP基础上加入了奖励函数和折扣因子，可以定义状态价值函数并通过贝尔曼方程求解。但它仍无决策者，状态转移是类似自然现象的随机过程。

马尔可夫决策过程（MDP）是三者中最完整的框架，在MRP基础上引入了动作空间和策略。奖励和转移概率同时依赖状态和动作，存在决策者（Agent）通过选择策略来最大化长期累积奖励。将策略固定后，MDP可减化为MRP，再用贝尔曼方程求解状态价值。